In [1]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
import time
warnings.filterwarnings("ignore")

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchaudio
from torchaudio import transforms as T

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from transformers import ASTForAudioClassification, ASTFeatureExtractor
import kagglehub
import wandb
from kaggle_secrets import UserSecretsClient
from torch.amp import autocast, GradScaler
from kaggle_secrets import UserSecretsClient

In [2]:
# ==========================================================
# MESSY MASHUP - AST CLASSIFIER
# Dynamic Stem Mixing + Noise Injection + Time Stretch
# Single File Solution for Kaggle with WandB Logging
# ==========================================================


# ===================== CONFIGURATION =====================

from kaggle_secrets import UserSecretsClient

# Get API key from Kaggle secrets
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login()

# Init run
wandb.init(
    project="milestone-5",
    name="very-final-submission",
    config={
        "features": "MFCC + Mel + Raw Audio",
        "models": ["LogisticRegression", "NaiveBayes", "CNN", "HuBERT", "AST"],
        "metric": "MacroF1"
    }
)

print("W&B initialized successfully")
# Paths
DATA_DIR = Path("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup")
STEMS_DIR = DATA_DIR / "genres_stems"
MASHUPS_DIR = DATA_DIR / "mashups"
NOISE_DIR = DATA_DIR / "ESC-50-master" / "audio"

# Audio Params
SR = 16000
CHUNK_DURATION = 10      # Seconds
OVERLAP_DURATION = 2    # Seconds
CHUNK_LEN = SR * CHUNK_DURATION
STEP = SR * (CHUNK_DURATION - OVERLAP_DURATION)

# Training Params
FOLDS = 4
EPOCHS = 20
BATCH_SIZE = 16
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Genres
GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
GENRE2IDX = {g:i for i,g in enumerate(GENRES)}
IDX2GENRE = {i:g for i,g in enumerate(GENRES)}

# Model Hub
MODEL_HUB_PATH = None 
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# ===================== GLOBAL CACHES (SPEED OPTIMIZATION) =====================

STEM_CACHE = {}  # Key: (genre, stem_type, song_id) -> waveform
NOISE_CACHE = []
CACHE_LOADED = False

# Organize stems by genre and stem_type for faster lookup
STEM_INDEX = {genre: {st: [] for st in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]} for genre in GENRES}

def preload_assets():
    """Loads stems and noise into RAM to avoid Disk I/O bottlenecks."""
    global STEM_CACHE, NOISE_CACHE, CACHE_LOADED, STEM_INDEX
    if CACHE_LOADED:
        return

    print(">> Preloading Stems into RAM...")
    start_time = time.time()
    
    # Load Stems
    for genre in GENRES:
        genre_path = STEMS_DIR / genre
        if not genre_path.exists():
            print(f"   Warning: {genre_path} not found")
            continue
        
        song_folders = list(genre_path.glob("*"))
        for song_folder in tqdm(song_folders, desc=f"Loading {genre}", leave=False):
            if not song_folder.is_dir():
                continue
            
            song_id = song_folder.name
            for stem_type in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]:
                p = song_folder / stem_type
                if p.exists():
                    try:
                        y, sr = torchaudio.load(p)
                        if sr != SR:
                            y = torchaudio.functional.resample(y, sr, SR)
                        y = y.mean(0)  # Convert to Mono
                        STEM_CACHE[(genre, stem_type, song_id)] = y
                        STEM_INDEX[genre][stem_type].append((song_id, y))
                    except Exception as e:
                        continue
    
    # Load Noise (ESC-50)
    print(">> Preloading Noise (ESC-50)...")
    if NOISE_DIR.exists():
        noise_files = list(NOISE_DIR.glob("*.wav"))[:50]  # Limit to 50 for RAM safety
        for p in tqdm(noise_files, desc="Loading Noise", leave=False):
            try:
                y, sr = torchaudio.load(p)
                if sr != SR:
                    y = torchaudio.functional.resample(y, sr, SR)
                y = y.mean(0)
                NOISE_CACHE.append(y)
            except Exception:
                continue
    else:
        print("   Warning: ESC-50 not found. Training without environmental noise.")

    CACHE_LOADED = True
    print(f">> Cache Ready. Stems: {len(STEM_CACHE)}, Noise Samples: {len(NOISE_CACHE)}")
    print(f">> Loading Time: {time.time() - start_time:.2f}s")

# ===================== AUDIO AUGMENTATION UTILS =====================

def get_random_stem(genre, stem_type, exclude_song_id=None):
    """
    Retrieves a random stem of specific type and genre from Cache.
    Returns: (waveform, song_id) or (None, None) if not found
    """
    candidates = STEM_INDEX.get(genre, {}).get(stem_type, [])
    
    if not candidates:
        return None, None
    
    # Filter out excluded song IDs
    if exclude_song_id:
        filtered = [(sid, wav) for sid, wav in candidates if sid != exclude_song_id]
        if filtered:
            candidates = filtered
    
    if not candidates:
        return None, None
        
    song_id, waveform = random.choice(candidates)
    return waveform.clone(), song_id

def apply_time_stretch(waveform, rate=1.0):
    """Simulates tempo change via resampling."""
    if rate == 1.0 or len(waveform) == 0:
        return waveform
    
    new_len = int(len(waveform) / rate)
    if new_len <= 0:
        return waveform
    
    indices = torch.linspace(0, len(waveform) - 1, new_len).long()
    stretched = waveform[indices]
    
    if len(stretched) < len(waveform):
        stretched = F.pad(stretched, (0, len(waveform) - len(stretched)))
    else:
        stretched = stretched[:len(waveform)]
        
    return stretched

def add_noise(waveform, snr_db=10):
    """Adds random environmental noise."""
    if len(NOISE_CACHE) == 0 or len(waveform) == 0:
        return waveform
        
    noise = random.choice(NOISE_CACHE)
    
    if len(noise) < len(waveform):
        repeats = int(np.ceil(len(waveform) / len(noise)))
        noise = noise.repeat(repeats)[:len(waveform)]
    else:
        start = random.randint(0, max(0, len(noise) - len(waveform)))
        noise = noise[start:start+len(waveform)]
        
    sig_power = waveform.pow(2).mean()
    noise_power = noise.pow(2).mean()
    
    if noise_power == 0 or sig_power == 0:
        return waveform
        
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    scale = torch.sqrt(target_noise_power / noise_power)
    
    noisy_waveform = waveform + (noise * scale)
    
    if noisy_waveform.abs().max() > 0:
        noisy_waveform = noisy_waveform / noisy_waveform.abs().max()
        
    return noisy_waveform

def create_augmented_mix(genre, target_length=CHUNK_LEN):
    """
    Dynamically creates a messy mix simulating test conditions.
    1. Picks 4 stems (drums, bass, vocals, others) from SAME genre.
    2. Mixes them.
    3. Applies Time Stretch.
    4. Applies Noise.
    """
    stems = []
    stem_types = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
    used_song_ids = set()
    
    for st in stem_types:
        stem_data, song_id = get_random_stem(genre, st, exclude_song_id=used_song_ids)
        if stem_data is not None:
            stems.append(stem_data)
            if song_id:
                used_song_ids.add(song_id)
            
    if len(stems) == 0:
        return None
        
    # Align lengths
    min_len = min(len(s) for s in stems)
    if min_len == 0:
        return None
    
    stems = [s[:min_len] for s in stems]
    mix = torch.stack(stems).sum(0)
    
    if mix.abs().max() > 0:
        mix = mix / mix.abs().max()
        
    # Augmentation 1: Time Stretch (Tempo)
    stretch_rate = random.uniform(0.9, 1.1)
    mix = apply_time_stretch(mix, stretch_rate)
    
    # Augmentation 2: Noise (SNR between 5dB and 20dB)
    snr = random.uniform(5, 20)
    mix = add_noise(mix, snr_db=snr)
    
    # Ensure target length
    if len(mix) < target_length:
        mix = F.pad(mix, (0, target_length - len(mix)))
    else:
        start = random.randint(0, max(0, len(mix) - target_length))
        mix = mix[start:start+target_length]
        
    return mix

# ===================== DATASET =====================

class MashupDataset(Dataset):
    def __init__(self, genres, labels, mode='train'):
        self.genres = genres
        self.labels = labels
        self.mode = mode
        self.len = len(genres)
        
    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        genre = self.genres[idx]
        label = self.labels[idx]
        
        # Generate Augmented Mix
        waveform = create_augmented_mix(genre)
        
        # Fallback if generation fails
        if waveform is None:
            waveform = torch.zeros(CHUNK_LEN)
            
        # Feature Extraction
        inputs = feature_extractor(
            waveform.numpy(),
            sampling_rate=SR,
            return_tensors="pt"
        )
        
        return inputs["input_values"].squeeze(0), torch.tensor(label)

# ===================== MODEL =====================

def build_model():
    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=10,
        ignore_mismatched_sizes=True
    )
    return model.to(DEVICE)

# ===================== TRAINING PIPELINE =====================

def train_pipeline():
    preload_assets()
    
    # Construct Training List (One entry per song folder to balance)
    train_genres = []
    train_labels = []
    
    song_folders = {}
    for genre in GENRES:
        song_folders[genre] = []
        g_path = STEMS_DIR / genre
        if g_path.exists():
            for f in g_path.glob("*"):
                if f.is_dir():
                    song_folders[genre].append(f)
                    
    for genre in GENRES:
        count = len(song_folders[genre])
        for _ in range(count):
            train_genres.append(genre)
            train_labels.append(GENRE2IDX[genre])
            
    print(f"Total Training Entries: {len(train_genres)}")
    
    if len(train_genres) == 0:
        print("ERROR: No training data found. Check STEMS_DIR path.")
        return
    
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
    saved_model_paths = []
    best_global_f1 = 0
    fold_f1_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_genres, train_labels)):
        print(f"\n========== Fold {fold} ==========")
        
        tr_genres = [train_genres[i] for i in tr_idx]
        tr_labels = [train_labels[i] for i in tr_idx]
        val_genres = [train_genres[i] for i in val_idx]
        val_labels = [train_labels[i] for i in val_idx]
        
        train_ds = MashupDataset(tr_genres, tr_labels, mode='train')
        val_ds = MashupDataset(val_genres, val_labels, mode='val')
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        
        model = build_model()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = GradScaler(device=DEVICE)
        
        best_f1 = 0
        
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            
            # Set Seed for Training Reproducibility
            torch.manual_seed(epoch + fold * 100)
            np.random.seed(epoch + fold * 100)
            random.seed(epoch + fold * 100)
            
            pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch+1}")
            for xb, yb in pbar:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                
                optimizer.zero_grad()
                
                with autocast(device_type=DEVICE):
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            scheduler.step()
            
            # Validation
            model.eval()
            # Fixed Seed for Stable Validation Metrics
            torch.manual_seed(42)
            np.random.seed(42)
            random.seed(42)
            
            preds, targets = [], []
            val_loss = 0
            
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE)
                    yb = yb.to(DEVICE)
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                    val_loss += loss.item()
                    preds.extend(outputs.logits.argmax(1).cpu().tolist())
                    targets.extend(yb.cpu().tolist())
            
            f1 = f1_score(targets, preds, average="macro")
            avg_train_loss = total_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            
            print(f"Fold {fold} | Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {f1:.4f}")
            
            wandb.log({
                "fold": fold,
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_f1_score": f1,
                "learning_rate": optimizer.param_groups[0]['lr']
            })
            
            if f1 > best_f1:
                best_f1 = f1
                path = f"model_fold{fold}.pth"
                torch.save(model.state_dict(), path)
                saved_model_paths.append(path)
                if f1 > best_global_f1:
                    best_global_f1 = f1
                
                wandb.log({
                    "fold": fold,
                    "best_f1": best_f1,
                    "best_epoch": epoch + 1
                })
                wandb.save(path)
                
        fold_f1_scores.append(best_f1)
        print(f"Fold {fold} Best F1: {best_f1:.4f}")
        
        wandb.log({f"fold_{fold}_best_f1": best_f1})

    # Log final summary
    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    
    print(f"\n{'='*50}")
    print(f"TRAINING COMPLETE")
    print(f"{'='*50}")
    print(f"Fold F1 Scores: {[f'{f:.4f}' for f in fold_f1_scores]}")
    print(f"Mean F1: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"Best Global F1: {best_global_f1:.4f}")
    
    wandb.log({
        "mean_f1_score": mean_f1,
        "std_f1_score": std_f1,
        "best_global_f1": best_global_f1,
        "fold_f1_scores": fold_f1_scores
    })
    
    for path in saved_model_paths:
        wandb.save(path)


# ===================== INFERENCE PIPELINE =====================

@torch.no_grad()
def inference_pipeline():
    print(">> Starting Inference Pipeline...")
    preload_assets()
    
    # Load Models
    models = []
    model_dir = "."
    
    try:
        model_dir = kagglehub.model_download(MODEL_HUB_PATH)
        print(f">> Models downloaded from: {model_dir}")
    except Exception as e:
        print(f">> Hub Download Failed. Using local models. {e}")
        
    for fold in range(FOLDS):
        model = build_model()
        path = f"{model_dir}/model_fold{fold}.pth"
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
            model.eval()
            models.append(model)
            print(f">> Loaded Fold {fold}")
        else:
            print(f">> Warning: Model {path} not found!")
            
    if len(models) == 0:
        print(">> No models loaded. Exiting.")
        return

    # Load Test Data
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
    
    submission_ids = sample["id"].tolist()
    
    # Map IDs to filenames
    if 'id' in test_df.columns:
        id_to_fname = dict(zip(test_df['id'], test_df['filename']))
        final_fnames = [id_to_fname.get(sid, None) for sid in submission_ids]
    else:
        final_fnames = test_df["filename"].tolist()

    preds = []
    
    for fname in tqdm(final_fnames, desc="Inference"):
        if fname is None:
            preds.append(GENRES[0])
            continue
            
        fpath = MASHUPS_DIR / Path(fname).name
        
        if not fpath.exists():
            preds.append(GENRES[0])
            continue
            
        try:
            mix, sr = torchaudio.load(fpath)
            if sr != SR:
                mix = torchaudio.functional.resample(mix, sr, SR)
            mix = mix.mean(0)
        except Exception:
            preds.append(GENRES[0])
            continue
            
        # Chunking
        if len(mix) < CHUNK_LEN:
            mix = F.pad(mix, (0, CHUNK_LEN - len(mix)))
            
        chunks = []
        length = len(mix)
        num_chunks = int(np.ceil((length - CHUNK_LEN) / STEP)) + 1
        num_chunks = min(num_chunks, 10)  # Cap for speed

        for i in range(num_chunks):
            start = i * STEP
            end = start + CHUNK_LEN
            chunk = mix[start:end]
            if len(chunk) < CHUNK_LEN:
                chunk = F.pad(chunk, (0, CHUNK_LEN - len(chunk)))
            chunks.append(chunk)
            
        # Ensemble Prediction
        logits_sum = 0
        total = 0
        
        for chunk in chunks:
            inputs = feature_extractor(
                chunk.numpy(),
                sampling_rate=SR,
                return_tensors="pt"
            )
            xb = inputs["input_values"].to(DEVICE)
            
            for model in models:
                outputs = model(input_values=xb)
                logits_sum += outputs.logits.cpu()
                total += 1
                
        if total > 0:
            logits_avg = logits_sum / total
            pred_idx = logits_avg.argmax(1).item()
            preds.append(IDX2GENRE[pred_idx])
        else:
            preds.append(GENRES[0])
            
    # Save Submission
    submission = pd.DataFrame({
        "id": submission_ids,
        "genre": preds
    })
    
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print(">> Submission saved to /kaggle/working/submission.csv")
    print(submission.head())
    
    genre_counts = pd.Series(preds).value_counts().to_dict()
    wandb.log({
        "total_predictions": len(preds),
        "genre_distribution": genre_counts
    })
    wandb.log_artifact("/kaggle/working/submission.csv", name="submission", type="submission")

# ===================== EXECUTION =====================

IS_TRAINING_MODE = True  # Set to False for Inference only

if __name__ == "__main__":
    start_time = time.time()
    if IS_TRAINING_MODE:
        train_pipeline()
    else:
        inference_pipeline()
    print(f">> Total Execution Time: {time.time() - start_time:.2f}s")
# =======================================================

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2002424 (23f2002424-shiv-nadar-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260331_040523-g5m87hec
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run very-final-submission
wandb: ⭐️ View project at https://wandb.ai/23f2002424-shiv-nadar-university/milestone-5
wandb: 🚀 View run at https://wandb.ai/23f2002424-shiv-nadar-university/milestone-5/runs/g5m87hec


W&B initialized successfully


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

>> Preloading Stems into RAM...


>> Preloading Noise (ESC-50)...


...

Fold 3 | Epoch 18 | Train Loss: 0.1839 | Val Loss: 0.1521 | F1: 0.9516


Fold 3 Epoch 19: 100%|██████████| 47/47 [01:01<00:00,  1.30s/it, loss=0.5362]


Fold 3 | Epoch 19 | Train Loss: 0.2005 | Val Loss: 0.1436 | F1: 0.9397


Fold 3 Epoch 20: 100%|██████████| 47/47 [01:01<00:00,  1.30s/it, loss=0.2248]


Fold 3 | Epoch 20 | Train Loss: 0.1712 | Val Loss: 0.1436 | F1: 0.9397
Fold 3 Best F1: 0.9516

TRAINING COMPLETE
Fold F1 Scores: ['0.9597', '0.9556', '0.9597', '0.9516']
Mean F1: 0.9566 ± 0.0034
Best Global F1: 0.9597
>> Total Execution Time: 7222.42s


In [3]:
# ==========================================================
# MESSY MASHUP - AST CLASSIFIER
# Dynamic Stem Mixing + Noise Injection + Time Stretch
# Single File Solution for Kaggle with WandB Logging
# ==========================================================


# ===================== CONFIGURATION =====================

from kaggle_secrets import UserSecretsClient

# Get API key from Kaggle secrets
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login()

# Init run
wandb.init(
    project="milestone-5",
    name="very-final-submission",
    config={
        "features": "MFCC + Mel + Raw Audio",
        "models": ["LogisticRegression", "NaiveBayes", "CNN", "HuBERT", "AST"],
        "metric": "MacroF1"
    }
)

print("W&B initialized successfully")
# Paths
DATA_DIR = Path("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup")
STEMS_DIR = DATA_DIR / "genres_stems"
MASHUPS_DIR = DATA_DIR / "mashups"
NOISE_DIR = DATA_DIR / "ESC-50-master" / "audio"

# Audio Params
SR = 16000
CHUNK_DURATION = 10      # Seconds
OVERLAP_DURATION = 2    # Seconds
CHUNK_LEN = SR * CHUNK_DURATION
STEP = SR * (CHUNK_DURATION - OVERLAP_DURATION)

# Training Params
FOLDS = 4
EPOCHS = 20
BATCH_SIZE = 16
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Genres
GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
GENRE2IDX = {g:i for i,g in enumerate(GENRES)}
IDX2GENRE = {i:g for i,g in enumerate(GENRES)}

# Model Hub
MODEL_HUB_PATH = None 
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)

# ===================== GLOBAL CACHES (SPEED OPTIMIZATION) =====================

STEM_CACHE = {}  # Key: (genre, stem_type, song_id) -> waveform
NOISE_CACHE = []
CACHE_LOADED = False

# Organize stems by genre and stem_type for faster lookup
STEM_INDEX = {genre: {st: [] for st in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]} for genre in GENRES}

def preload_assets():
    """Loads stems and noise into RAM to avoid Disk I/O bottlenecks."""
    global STEM_CACHE, NOISE_CACHE, CACHE_LOADED, STEM_INDEX
    if CACHE_LOADED:
        return

    print(">> Preloading Stems into RAM...")
    start_time = time.time()
    
    # Load Stems
    for genre in GENRES:
        genre_path = STEMS_DIR / genre
        if not genre_path.exists():
            print(f"   Warning: {genre_path} not found")
            continue
        
        song_folders = list(genre_path.glob("*"))
        for song_folder in tqdm(song_folders, desc=f"Loading {genre}", leave=False):
            if not song_folder.is_dir():
                continue
            
            song_id = song_folder.name
            for stem_type in ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]:
                p = song_folder / stem_type
                if p.exists():
                    try:
                        y, sr = torchaudio.load(p)
                        if sr != SR:
                            y = torchaudio.functional.resample(y, sr, SR)
                        y = y.mean(0)  # Convert to Mono
                        STEM_CACHE[(genre, stem_type, song_id)] = y
                        STEM_INDEX[genre][stem_type].append((song_id, y))
                    except Exception as e:
                        continue
    
    # Load Noise (ESC-50)
    print(">> Preloading Noise (ESC-50)...")
    if NOISE_DIR.exists():
        noise_files = list(NOISE_DIR.glob("*.wav"))[:50]  # Limit to 50 for RAM safety
        for p in tqdm(noise_files, desc="Loading Noise", leave=False):
            try:
                y, sr = torchaudio.load(p)
                if sr != SR:
                    y = torchaudio.functional.resample(y, sr, SR)
                y = y.mean(0)
                NOISE_CACHE.append(y)
            except Exception:
                continue
    else:
        print("   Warning: ESC-50 not found. Training without environmental noise.")

    CACHE_LOADED = True
    print(f">> Cache Ready. Stems: {len(STEM_CACHE)}, Noise Samples: {len(NOISE_CACHE)}")
    print(f">> Loading Time: {time.time() - start_time:.2f}s")

# ===================== AUDIO AUGMENTATION UTILS =====================

def get_random_stem(genre, stem_type, exclude_song_id=None):
    """
    Retrieves a random stem of specific type and genre from Cache.
    Returns: (waveform, song_id) or (None, None) if not found
    """
    candidates = STEM_INDEX.get(genre, {}).get(stem_type, [])
    
    if not candidates:
        return None, None
    
    # Filter out excluded song IDs
    if exclude_song_id:
        filtered = [(sid, wav) for sid, wav in candidates if sid != exclude_song_id]
        if filtered:
            candidates = filtered
    
    if not candidates:
        return None, None
        
    song_id, waveform = random.choice(candidates)
    return waveform.clone(), song_id

def apply_time_stretch(waveform, rate=1.0):
    """Simulates tempo change via resampling."""
    if rate == 1.0 or len(waveform) == 0:
        return waveform
    
    new_len = int(len(waveform) / rate)
    if new_len <= 0:
        return waveform
    
    indices = torch.linspace(0, len(waveform) - 1, new_len).long()
    stretched = waveform[indices]
    
    if len(stretched) < len(waveform):
        stretched = F.pad(stretched, (0, len(waveform) - len(stretched)))
    else:
        stretched = stretched[:len(waveform)]
        
    return stretched

def add_noise(waveform, snr_db=10):
    """Adds random environmental noise."""
    if len(NOISE_CACHE) == 0 or len(waveform) == 0:
        return waveform
        
    noise = random.choice(NOISE_CACHE)
    
    if len(noise) < len(waveform):
        repeats = int(np.ceil(len(waveform) / len(noise)))
        noise = noise.repeat(repeats)[:len(waveform)]
    else:
        start = random.randint(0, max(0, len(noise) - len(waveform)))
        noise = noise[start:start+len(waveform)]
        
    sig_power = waveform.pow(2).mean()
    noise_power = noise.pow(2).mean()
    
    if noise_power == 0 or sig_power == 0:
        return waveform
        
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    scale = torch.sqrt(target_noise_power / noise_power)
    
    noisy_waveform = waveform + (noise * scale)
    
    if noisy_waveform.abs().max() > 0:
        noisy_waveform = noisy_waveform / noisy_waveform.abs().max()
        
    return noisy_waveform

def create_augmented_mix(genre, target_length=CHUNK_LEN):
    """
    Dynamically creates a messy mix simulating test conditions.
    1. Picks 4 stems (drums, bass, vocals, others) from SAME genre.
    2. Mixes them.
    3. Applies Time Stretch.
    4. Applies Noise.
    """
    stems = []
    stem_types = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
    used_song_ids = set()
    
    for st in stem_types:
        stem_data, song_id = get_random_stem(genre, st, exclude_song_id=used_song_ids)
        if stem_data is not None:
            stems.append(stem_data)
            if song_id:
                used_song_ids.add(song_id)
            
    if len(stems) == 0:
        return None
        
    # Align lengths
    min_len = min(len(s) for s in stems)
    if min_len == 0:
        return None
    
    stems = [s[:min_len] for s in stems]
    mix = torch.stack(stems).sum(0)
    
    if mix.abs().max() > 0:
        mix = mix / mix.abs().max()
        
    # Augmentation 1: Time Stretch (Tempo)
    stretch_rate = random.uniform(0.9, 1.1)
    mix = apply_time_stretch(mix, stretch_rate)
    
    # Augmentation 2: Noise (SNR between 5dB and 20dB)
    snr = random.uniform(5, 20)
    mix = add_noise(mix, snr_db=snr)
    
    # Ensure target length
    if len(mix) < target_length:
        mix = F.pad(mix, (0, target_length - len(mix)))
    else:
        start = random.randint(0, max(0, len(mix) - target_length))
        mix = mix[start:start+target_length]
        
    return mix

# ===================== DATASET =====================

class MashupDataset(Dataset):
    def __init__(self, genres, labels, mode='train'):
        self.genres = genres
        self.labels = labels
        self.mode = mode
        self.len = len(genres)
        
    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        genre = self.genres[idx]
        label = self.labels[idx]
        
        # Generate Augmented Mix
        waveform = create_augmented_mix(genre)
        
        # Fallback if generation fails
        if waveform is None:
            waveform = torch.zeros(CHUNK_LEN)
            
        # Feature Extraction
        inputs = feature_extractor(
            waveform.numpy(),
            sampling_rate=SR,
            return_tensors="pt"
        )
        
        return inputs["input_values"].squeeze(0), torch.tensor(label)

# ===================== MODEL =====================

def build_model():
    model = ASTForAudioClassification.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593",
        num_labels=10,
        ignore_mismatched_sizes=True
    )
    return model.to(DEVICE)

# ===================== TRAINING PIPELINE =====================

def train_pipeline():
    preload_assets()
    
    # Construct Training List (One entry per song folder to balance)
    train_genres = []
    train_labels = []
    
    song_folders = {}
    for genre in GENRES:
        song_folders[genre] = []
        g_path = STEMS_DIR / genre
        if g_path.exists():
            for f in g_path.glob("*"):
                if f.is_dir():
                    song_folders[genre].append(f)
                    
    for genre in GENRES:
        count = len(song_folders[genre])
        for _ in range(count):
            train_genres.append(genre)
            train_labels.append(GENRE2IDX[genre])
            
    print(f"Total Training Entries: {len(train_genres)}")
    
    if len(train_genres) == 0:
        print("ERROR: No training data found. Check STEMS_DIR path.")
        return
    
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
    saved_model_paths = []
    best_global_f1 = 0
    fold_f1_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_genres, train_labels)):
        print(f"\n========== Fold {fold} ==========")
        
        tr_genres = [train_genres[i] for i in tr_idx]
        tr_labels = [train_labels[i] for i in tr_idx]
        val_genres = [train_genres[i] for i in val_idx]
        val_labels = [train_labels[i] for i in val_idx]
        
        train_ds = MashupDataset(tr_genres, tr_labels, mode='train')
        val_ds = MashupDataset(val_genres, val_labels, mode='val')
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        
        model = build_model()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = GradScaler(device=DEVICE)
        
        best_f1 = 0
        
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            
            # Set Seed for Training Reproducibility
            torch.manual_seed(epoch + fold * 100)
            np.random.seed(epoch + fold * 100)
            random.seed(epoch + fold * 100)
            
            pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch+1}")
            for xb, yb in pbar:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                
                optimizer.zero_grad()
                
                with autocast(device_type=DEVICE):
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            scheduler.step()
            
            # Validation
            model.eval()
            # Fixed Seed for Stable Validation Metrics
            torch.manual_seed(42)
            np.random.seed(42)
            random.seed(42)
            
            preds, targets = [], []
            val_loss = 0
            
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(DEVICE)
                    yb = yb.to(DEVICE)
                    outputs = model(input_values=xb)
                    loss = F.cross_entropy(outputs.logits, yb)
                    val_loss += loss.item()
                    preds.extend(outputs.logits.argmax(1).cpu().tolist())
                    targets.extend(yb.cpu().tolist())
            
            f1 = f1_score(targets, preds, average="macro")
            avg_train_loss = total_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            
            print(f"Fold {fold} | Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {f1:.4f}")
            
            wandb.log({
                "fold": fold,
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_f1_score": f1,
                "learning_rate": optimizer.param_groups[0]['lr']
            })
            
            if f1 > best_f1:
                best_f1 = f1
                path = f"model_fold{fold}.pth"
                torch.save(model.state_dict(), path)
                saved_model_paths.append(path)
                if f1 > best_global_f1:
                    best_global_f1 = f1
                
                wandb.log({
                    "fold": fold,
                    "best_f1": best_f1,
                    "best_epoch": epoch + 1
                })
                wandb.save(path)
                
        fold_f1_scores.append(best_f1)
        print(f"Fold {fold} Best F1: {best_f1:.4f}")
        
        wandb.log({f"fold_{fold}_best_f1": best_f1})

    # Log final summary
    mean_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)
    
    print(f"\n{'='*50}")
    print(f"TRAINING COMPLETE")
    print(f"{'='*50}")
    print(f"Fold F1 Scores: {[f'{f:.4f}' for f in fold_f1_scores]}")
    print(f"Mean F1: {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"Best Global F1: {best_global_f1:.4f}")
    
    wandb.log({
        "mean_f1_score": mean_f1,
        "std_f1_score": std_f1,
        "best_global_f1": best_global_f1,
        "fold_f1_scores": fold_f1_scores
    })
    
    for path in saved_model_paths:
        wandb.save(path)


# ===================== INFERENCE PIPELINE =====================

@torch.no_grad()
def inference_pipeline():
    print(">> Starting Inference Pipeline...")
    preload_assets()
    
    # Load Models
    models = []
    model_dir = "."
    
    try:
        model_dir = kagglehub.model_download(MODEL_HUB_PATH)
        print(f">> Models downloaded from: {model_dir}")
    except Exception as e:
        print(f">> Hub Download Failed. Using local models. {e}")
        
    for fold in range(FOLDS):
        model = build_model()
        path = f"{model_dir}/model_fold{fold}.pth"
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
            model.eval()
            models.append(model)
            print(f">> Loaded Fold {fold}")
        else:
            print(f">> Warning: Model {path} not found!")
            
    if len(models) == 0:
        print(">> No models loaded. Exiting.")
        return

    # Load Test Data
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
    
    submission_ids = sample["id"].tolist()
    
    # Map IDs to filenames
    if 'id' in test_df.columns:
        id_to_fname = dict(zip(test_df['id'], test_df['filename']))
        final_fnames = [id_to_fname.get(sid, None) for sid in submission_ids]
    else:
        final_fnames = test_df["filename"].tolist()

    preds = []
    
    for fname in tqdm(final_fnames, desc="Inference"):
        if fname is None:
            preds.append(GENRES[0])
            continue
            
        fpath = MASHUPS_DIR / Path(fname).name
        
        if not fpath.exists():
            preds.append(GENRES[0])
            continue
            
        try:
            mix, sr = torchaudio.load(fpath)
            if sr != SR:
                mix = torchaudio.functional.resample(mix, sr, SR)
            mix = mix.mean(0)
        except Exception:
            preds.append(GENRES[0])
            continue
            
        # Chunking
        if len(mix) < CHUNK_LEN:
            mix = F.pad(mix, (0, CHUNK_LEN - len(mix)))
            
        chunks = []
        length = len(mix)
        num_chunks = int(np.ceil((length - CHUNK_LEN) / STEP)) + 1
        num_chunks = min(num_chunks, 10)  # Cap for speed

        for i in range(num_chunks):
            start = i * STEP
            end = start + CHUNK_LEN
            chunk = mix[start:end]
            if len(chunk) < CHUNK_LEN:
                chunk = F.pad(chunk, (0, CHUNK_LEN - len(chunk)))
            chunks.append(chunk)
            
        # Ensemble Prediction
        logits_sum = 0
        total = 0
        
        for chunk in chunks:
            inputs = feature_extractor(
                chunk.numpy(),
                sampling_rate=SR,
                return_tensors="pt"
            )
            xb = inputs["input_values"].to(DEVICE)
            
            for model in models:
                outputs = model(input_values=xb)
                logits_sum += outputs.logits.cpu()
                total += 1
                
        if total > 0:
            logits_avg = logits_sum / total
            pred_idx = logits_avg.argmax(1).item()
            preds.append(IDX2GENRE[pred_idx])
        else:
            preds.append(GENRES[0])
            
    # Save Submission
    submission = pd.DataFrame({
        "id": submission_ids,
        "genre": preds
    })
    
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print(">> Submission saved to /kaggle/working/submission.csv")
    print(submission.head())
    
    genre_counts = pd.Series(preds).value_counts().to_dict()
    wandb.log({
        "total_predictions": len(preds),
        "genre_distribution": genre_counts
    })
    wandb.log_artifact("/kaggle/working/submission.csv", name="submission", type="submission")

# ===================== EXECUTION =====================

IS_TRAINING_MODE = False  # Set to False for Inference only

if __name__ == "__main__":
    start_time = time.time()
    if IS_TRAINING_MODE:
        train_pipeline()
    else:
        inference_pipeline()
    print(f">> Total Execution Time: {time.time() - start_time:.2f}s")
# =======================================================

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading history steps 130-132, summary, console lines 213-224
wandb: 
wandb: Run history:
wandb:     best_epoch ▁▁▂▃▃▅▅▇▇▁▂▂▃▃▄▅▆▆▇█▁▁▂▂▃▄▆▆▆▇▁▁▂▂▃▄▅▆▆█
wandb:        best_f1 ▁▃▅▅▆▇▇██▁▄▅▆▆▆▇▇███▂▃▄▄▅▇▇███▂▄▅▅▆▆▇▇██
wandb: best_global_f1 ▁
wandb:          epoch ▁▂▄▅▅▆▇▇█▁▂▃▃▃▄▆▆▇██▁▁▂▃▃▄▅▅▆▆▁▂▂▃▃▅▅▆▆▇
wandb:           fold ▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆▆▆▆▆▆████████
wandb: fold_0_best_f1 ▁
wandb: fold_1_best_f1 ▁
wandb: fold_2_best_f1 ▁
wandb: fold_3_best_f1 ▁
wandb:  learning_rate ██▇▆▅▄▃▂▁▁██▇▆▆▄▃▂▂▁▇▇▆▅▃▂▁▁▁▁█▇▇▅▄▃▂▂▁▁
wandb:             +5 ...
wandb: 
wandb: Run summary:
wandb:     best_epoch 18
wandb:        best_f1 0.95161
wandb: best_global_f1 0.95975
wandb:          epoch 20
wandb:           fold 3
wandb: fold_0_best_f1 0.95966
wandb: fold_1_best_f1 0.95557
wandb: fold_2_best_f1 0.95975
wandb: fold_3_best_f1

W&B initialized successfully
>> Starting Inference Pipeline...
>> Preloading Stems into RAM...


>> Preloading Noise (ESC-50)...


>> Cache Ready. Stems: 3000, Noise Samples: 50
>> Loading Time: 358.94s
>> Hub Download Failed. Using local models. 'NoneType' object has no attribute 'split'


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


>> Loaded Fold 3


Inference: 100%|██████████| 3020/3020 [1:17:09<00:00,  1.53s/it]


>> Submission saved to /kaggle/working/submission.csv
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
>> Total Execution Time: 4993.39s
